# llmon vs Ollama — side-by-side speed test (Colab T4)

Installs both engines, serves the **same Qwen2.5-0.5B weights** in each,
and compares tokens/second. Run top to bottom (~15 min, mostly the
one-time CUDA build).

In [ ]:
%%bash
set -e
nvidia-smi --query-gpu=name --format=csv,noheader  # expect Tesla T4
command -v ollama >/dev/null || curl -fsSL https://ollama.com/install.sh | sh
export PATH="/usr/local/bin:$PATH"
nohup ollama serve >/tmp/ollama.log 2>&1 &
sleep 5
ollama pull qwen2.5:0.5b
ollama list

In [ ]:
%%bash
set -e
export PATH="$HOME/.cargo/bin:$HOME/.local/bin:$PATH"
command -v cargo >/dev/null || curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal
test -d /content/llmon || git clone --depth 1 https://github.com/deepsuthar496/llmon /content/llmon
cd /content/llmon && cargo build --release --locked
mkdir -p "$HOME/.local/bin" && ln -sf /content/llmon/target/release/llmon "$HOME/.local/bin/llmon"
sudo apt-get update -qq && sudo apt-get install -y -qq cmake g++ >/dev/null
cd /content/llmon && ./scripts/build-backend.sh --cuda
export LD_LIBRARY_PATH="$HOME/.local/lib:$LD_LIBRARY_PATH"
llmon pull Qwen/Qwen2.5-0.5B-Instruct-GGUF/qwen2.5-0.5b-instruct-q4_k_m.gguf
nohup llmon serve >/tmp/llmon.log 2>&1 &
sleep 3 && curl -s localhost:11435/health && echo "  <- llmon up"

## Race — same prompts, 60 tokens each, median of 3

In [ ]:
import json, statistics, time, urllib.request

PROMPTS = ["Summarize the water cycle in one paragraph",
           "Explain what a prime number is in one paragraph",
           "Describe the solar system in one paragraph"]
ENGINES = {"ollama": ("http://localhost:11434/api/generate", "qwen2.5:0.5b"),
           "llmon": ("http://localhost:11435/api/generate",
                     "Qwen/Qwen2.5-0.5B-Instruct-GGUF/qwen2.5-0.5b-instruct-q4_k_m.gguf")}

def bench(url, model):
    totals = []
    for p in ["warmup"] + PROMPTS:
        body = json.dumps({"model": model, "prompt": p, "stream": True,
                           "options": {"num_predict": 60, "temperature": 0}}).encode()
        req = urllib.request.Request(url, data=body, headers={"Content-Type": "application/json"})
        t0, first, toks = time.monotonic(), None, 0
        for line in urllib.request.urlopen(req, timeout=300):
            line = line.strip()
            if not line:
                continue
            try:
                c = json.loads(line)
            except ValueError:
                continue
            if first is None:
                first = time.monotonic() - t0
            if c.get("done"):
                toks = c.get("eval_count", 60) or 60
                break
        if p != "warmup":
            totals.append((first or 0, time.monotonic() - t0, toks))
    ttft = statistics.median(t[0] for t in totals)
    total = statistics.median(t[1] for t in totals)
    return ttft, total, 60 / total

results = {}
for name, (url, model) in ENGINES.items():
    results[name] = bench(url, model)
    print(f"{name}: done", flush=True)

print("\n  engine  |  first token  |  60 tokens  |  tok/s")
print("  " + "-" * 50)
for name, (ttft, total, rate) in results.items():
    print(f"  {name:<7} |  {ttft:>6.2f}s     |  {total:>6.2f}s   |  {rate:>5.1f}")
o, l = results["ollama"][2], results["llmon"][2]
print(f"\n  Winner: {'llmon' if l > o else 'ollama'} ({max(l, o) / min(l, o):.2f}x)" if o and l else "")

import matplotlib.pyplot as plt
plt.barh(list(results), [v[2] for v in results.values()])
plt.xlabel("tokens / second (higher is better)")
plt.title("llmon vs Ollama — same Qwen2.5-0.5B weights, Colab T4")
plt.show()

Done. Same kernels → expect a close race; llmon usually pulls ahead on
repeated prompts (ngram speculative decoding). Paste the table + chart
into a llmon GitHub issue along with your `nvidia-smi` output.